### Kita mulai dari Sentiment Analysis dasar menggunakan NLP

In [1]:
#  Sentiment Analysis dengan NLP dan KNN menggunakan Sastrawi dan TF-IDF
# Import Libarary

import pandas as pd
import numpy as np
import re 
import string

from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
import pickle ## biasanya untuk menyimpan model

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, f1_score, accuracy_score, precision_score, recall_score


In [2]:
# Inisialisasi Sastrawi Stopword Remover dan Stemmer
factory = StopWordRemoverFactory()
stopword_remover = factory.create_stop_word_remover()
stemmer_factory = StemmerFactory()
stemmer = stemmer_factory.create_stemmer()


In [3]:
# Load Dataset from github
url = "https://raw.githubusercontent.com/Muhammad-Ikhwan-Fathulloh/Advanced-Machine-Learning-Course/refs/heads/main/KNN/Datasets/sentiment_cellular.csv"

data = pd.read_csv(url, encoding='latin-1')

In [4]:
data['Sentiment'].value_counts().reset_index()

,Sentiment,count
0,negative,161
1,positive,139


In [4]:
data.head(5)

,Sentiment,Text Tweet
0,positive,<USER_MENTION> #BOIKOT_<PROVIDER_NAME> Gunakan...
1,positive,"Saktinya balik lagi, alhamdulillah :v <PROVIDE..."
2,negative,Selamat pagi <PROVIDER_NAME> bisa bantu kenap...
3,negative,Dear <PROVIDER_NAME> akhir2 ini jaringan data ...
4,negative,Selamat malam PENDUSTA <PROVIDER_NAME>


In [5]:
# Coba selesaikan masalah singkatan singkatan dalam permasalahan NLP

chat_words = {"IMHO": "In My Humble Opinion","ASAP": "As Soon As Possible","FYI": "For Your Information","BFF": "Best Friends Forever",
              "WYD": "What You Doing","YOLO": "You Only Live Once"}
contoh_kata = "IMHO, people should do the task ASAP, because in this live YOLO"


def konversi_teks(text: str) -> str:
    clean_text = re.sub(r'[^\w\s]', '', text)
    new_text = []
    for i in clean_text.split():
        if i.upper() in chat_words.keys():
            new_text.append(chat_words[i.upper()])
        else:
            new_text.append(i)
    return " ".join(new_text)

konversi_teks(contoh_kata)

'In My Humble Opinion people should do the task As Soon As Possible because in this live You Only Live Once'

In [6]:
## MULAII PREPROCESSING
# 1. Lowercase seluruh dataset
def lower_cleansing(text):
    text = re.sub(r'[^\w\s!?]', '', text) # Hapus punctuation tapi disisakan ! dan ?, karena case sentiment
    text = re.sub(r'http\S+|www\.\S+', '', text) # Hapus URL
    text = re.sub(r'@\w+', '', text)              # mention
    text = re.sub(r'#', '', text)                 # hashtag symbol
    text = text.replace('_', ' ')   # ganti underskor jadi space
    text = re.sub(r'\d+', '', text) # Hapus Angka
    text = re.sub(r'\s+', ' ', text).strip() # rapikan spasi
    text = text.lower()
    
    return text
data['Text Tweet'] = data['Text Tweet'].apply(lower_cleansing)
data.head(5)

,Sentiment,Text Tweet
0,positive,user mention boikot provider name gunakan prod...
1,positive,saktinya balik lagi alhamdulillah v provider name
2,negative,selamat pagi provider name bisa bantu kenapa d...
3,negative,dear provider name akhir ini jaringan data lem...
4,negative,selamat malam pendusta provider name


In [7]:
## Tokenisasi dan Stopwords
# Untu belajar dasarnya kita tokenisasi dengan yang dasar dulu pakai split
# Stopword disini kita tetap mempertahankan beberapa kata negasi, karena bisa saja memiliki bobot tinggi dalam
# Suatu kalimat
keep_words = {"tidak", "bukan", "belum", "jangan", "tak", "tanpa"}
stopwords = set(factory.get_stop_words())
stopwords = stopwords - keep_words


def tokenization(text):
    text = text.split()
    text = [t for t in text if t not in stopwords]
    return text
tokens = data["Text Tweet"].apply(tokenization)
tokens


0      [user, mention, boikot, provider, name, gunaka...
1      [saktinya, balik, alhamdulillah, v, provider, ...
2      [selamat, pagi, provider, name, bantu, kamar, ...
3      [dear, provider, name, akhir, jaringan, data, ...
4             [selamat, malam, pendusta, provider, name]
                             ...                        
295    [pantesan, lancar, sinyal, provider, name, g, ...
296       [alhamdulillah, lancar, pakai, provider, name]
297    [untung, pakai, internet, provider, name, lanc...
298    [tempat, ramai, lokasi, wisata, provider, name...
299    [sinyal, provider, name, amsyong, d, stadion, ...
Name: Text Tweet, Length: 300, dtype: object

In [ ]:
# Melakukan Stemming
def stemming(tokens):
    text = [stemmer.stem(token) for token in tokens]
    return " ".join(text) # kita satukan lagi menjadi string karena kita ingin mengembedding menggunakan TFIDF

stem_res = tokens.apply(stemming)
print(stem_res)

0      user mention boikot provider name guna produk ...
1              sakti balik alhamdulillah v provider name
2      selamat pagi provider name bantu kamar sinyal ...
3      dear provider name akhir jaring data lot bange...
4                      selamat malam dusta provider name
                             ...                        
295    pantesan lancar sinyal provider name g lancar ...
296             alhamdulillah lancar pakai provider name
297    untung pakai internet provider name lancar jad...
298    tempat ramai lokasi wisata provider name tetap...
299      sinyal provider name amsyong d stadion gajayana
Name: Text Tweet, Length: 300, dtype: str


In [13]:
# Vectorization menggunakan TF IDF
# Split data ke fitur dan target
tfidf_vectorizer = TfidfVectorizer()
X = tfidf_vectorizer.fit_transform(stem_res)
y = data["Sentiment"]

# Coba buat dataset isi vectorizer
tfidf_df = pd.DataFrame(X.toarray(), columns= tfidf_vectorizer.get_feature_names_out())
tfidf_df.head(5)



,acara,aceh,ada,adhan,aja,ajaib,ajar,akhir,akses,aksi,...,ya,yaa,yah,yess,yg,youtube,youtubenya,youtubeyondergenflix,yuk,zalim
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.395515,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [14]:
## Split dataset ke train dan test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## Inisiasi model KNN

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)

# Predictions
y_pred = knn.predict(X_test)


In [15]:
# Melihat hasil Evaluasi
print("\nClassification Report:\n", classification_report(y_test, y_pred))


Classification Report:
               precision    recall  f1-score   support

    negative       0.83      0.73      0.77        33
    positive       0.71      0.81      0.76        27

    accuracy                           0.77        60
   macro avg       0.77      0.77      0.77        60
weighted avg       0.77      0.77      0.77        60



In [18]:
# Coba pakai gridsearch
from sklearn.model_selection import GridSearchCV

knn_params = {
    'n_neighbors' :[3, 5, 7, 10, 12],
    'weights' : ['uniform', 'distance'],
    'metric': ["euclidean", 'cosine']
    
}

# setup GridSearch
grid_search = GridSearchCV(knn, param_grid=knn_params,cv = 5, scoring='accuracy', n_jobs=-1 )

grid_search.fit(X_train, y_train)
print(f"Best Parameters: {grid_search.best_params_}")

Best Parameters: {'metric': 'cosine', 'n_neighbors': 7, 'weights': 'distance'}


In [22]:
y_pred_grid = grid_search.predict(X_test)
print(f"hasil evaluasi \n {classification_report(y_test, y_pred_grid)}")

hasil evaluasi 
               precision    recall  f1-score   support

    negative       0.81      0.88      0.84        33
    positive       0.83      0.74      0.78        27

    accuracy                           0.82        60
   macro avg       0.82      0.81      0.81        60
weighted avg       0.82      0.82      0.82        60

